## Task 3 — LLM Categorization (few-shot + MCC fallback + disk cache)

Reads `artifacts/transactions_enriched.json` (NDJSON, all users) and produces a
transaction -> category mapping used by the analytics/prediction layers.

**Business constraints -> design decisions**

| Constraint | Decision in this notebook |
|---|---|
| No labeled training data | Few-shot prompt over a fixed taxonomy; deterministic MCC mapping as the reference/fallback |
| Strict cost limits | Categorize **unique merchant profiles**, not raw transactions; batch requests; disk cache; hard `MAX_LLM_CALLS` cap |
| Limited ML expertise | Off-the-shelf chat completion API, no fine-tuning |
| Must never block time-to-first-insight | LLM failure / missing key / cost cap degrades to deterministic MCC mapping — the notebook still completes with 100% coverage |

**Why profiles instead of transactions:** a category depends only on the merchant and
MCC fields, not on the amount or timestamp. Client 1696 has ~30.7k transactions but only a
few hundred distinct merchant/MCC combinations, so profile-level categorization cuts LLM
calls by orders of magnitude while producing an identical per-transaction result.

**Inputs (read-only):** `artifacts/transactions_enriched.json`

**Outputs (under `artifacts/`):**
- `transaction_categories{_client}.jsonl` — one row per transaction (`id`, `category`, `source`, `confidence`)
- `merchant_profiles{_client}.json` — the distinct profiles that were categorized
- `categorization_report{_client}.json` — QA: coverage, source mix, MCC agreement, cost counters

**Required env (`.env` in project root):** `LLM_API_KEY` (or `OPENAI_API_KEY`), `LLM_MODEL`, and `OPENAI_BASE_URL` (or `LLM_API_BASE`) when using a gateway/proxy token


## Imports and project-root bootstrap

Notebooks run with their own directory as CWD, so put the project root on `sys.path` before importing `data_processing.*` / `model.*`.


In [14]:
from __future__ import annotations

import hashlib
import json
import os
import statistics
import sys
from collections import Counter
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the folder containing `data/` and `artifacts/`."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing 'data/' and 'artifacts/'")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from diskcache import Cache
from dotenv import load_dotenv

from data_processing.categorize_core import (
    DEFAULT_TAXONOMY,
    categorize_with_cache_and_fallback,
    make_openai_llm_call,
    transaction_fingerprint,
)
from model.analytics_core import mcc_to_category

print("Project root:", ROOT)
print("Taxonomy:", list(DEFAULT_TAXONOMY))


Project root: /Users/nicholasp/Personal Coding/JHU/personal finance
Taxonomy: ['Dining', 'Groceries', 'Utilities', 'Transportation', 'Entertainment', 'Shopping', 'Travel', 'Housing', 'Healthcare', 'Education', 'Income', 'Transfers', 'Subscriptions', 'Fees & Interest', 'Cash Withdrawal', 'Other/Uncategorized']


## Config

- `CLIENT_ID_FILTER = 1696` keeps the graded demo run cheap; set to `None` to categorize every user.
- `MAX_LLM_CALLS` is the hard cost guardrail. Once it trips, remaining profiles are categorized
  by deterministic MCC mapping and the report records the degradation.
- `CACHE_VERSION` must match the default in `categorize_core.categorize_with_cache_and_fallback`,
  because this notebook computes the same cache keys to evict degraded entries.


In [15]:
load_dotenv(ROOT / ".env")

CLIENT_ID_FILTER: int | None = 1696   # demo user for submission; None = all users (expensive)
BATCH_SIZE = 40                       # merchant profiles per LLM request
CONF_THRESHOLD = 0.6                  # below this, trust the deterministic MCC mapping instead
MAX_LLM_CALLS: int | None = 1         # hard cost cap; remaining profiles fall back to MCC
CACHE_VERSION = "v1"                  # must match categorize_core's default cache_version

LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
API_KEY = os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

ARTIFACTS = ROOT / "artifacts"
NDJSON_PATH = ARTIFACTS / "transactions_enriched.json"
CACHE_DIR = ARTIFACTS / "llm_cache"

SUFFIX = f"_{CLIENT_ID_FILTER}" if CLIENT_ID_FILTER is not None else ""
OUT_CATEGORIES = ARTIFACTS / f"transaction_categories{SUFFIX}.jsonl"
OUT_PROFILES = ARTIFACTS / f"merchant_profiles{SUFFIX}.json"
OUT_REPORT = ARTIFACTS / f"categorization_report{SUFFIX}.json"

if not NDJSON_PATH.exists():
    raise FileNotFoundError(
        f"Missing {NDJSON_PATH}. Run data_processing/clean.ipynb first."
    )

print("NDJSON:      ", NDJSON_PATH)
print("Cache:       ", CACHE_DIR)
print("Categories:  ", OUT_CATEGORIES)
print("Profiles:    ", OUT_PROFILES)
print("Report:      ", OUT_REPORT)
print("Model:       ", LLM_MODEL)
print("Base URL:    ", BASE_URL or "(default OpenAI)")
print("Client filter:", CLIENT_ID_FILTER if CLIENT_ID_FILTER is not None else "ALL USERS")
print("API key found:", bool(API_KEY))


NDJSON:       /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched.json
Cache:        /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/llm_cache
Categories:   /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transaction_categories_1696.jsonl
Profiles:     /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/merchant_profiles_1696.json
Report:       /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/categorization_report_1696.json
Model:        gpt-5.2
Base URL:     https://aibe.mygreatlearning.com/openai/v1
Client filter: 1696
API key found: True


## Pass 1 — stream the cleaned NDJSON and build distinct merchant profiles

The NDJSON is ~600 MB, so it is streamed line by line (never loaded into a DataFrame).
A cheap substring pre-filter skips most lines when a single client is selected; the
authoritative check still happens after parsing.

Each profile carries only non-PII merchant/MCC fields, and its `id` is a stable hash of
those fields — so the disk cache is reused across reruns **and across users**.


In [16]:
PROFILE_FIELDS = (
    "mcc_code",
    "mcc_description",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "is_online",
)


def profile_key(tx: dict) -> str:
    """Merchant/MCC identity of a transaction (amount and date are irrelevant to category)."""
    return "|".join("" if tx.get(f) is None else str(tx.get(f)) for f in PROFILE_FIELDS)


def profile_id(key: str) -> str:
    """Stable id so cache keys survive reruns and reordering."""
    return "p" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]


def iter_transactions(path: Path, client_id: int | None):
    """Yield cleaned transaction dicts, optionally filtered to one client."""
    needle = None if client_id is None else f'"{client_id}"'
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if needle is not None and needle not in line:
                continue  # fast pre-filter only; exact check below
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if client_id is not None and str(obj.get("client_id")) != str(client_id):
                continue
            yield obj


profiles: dict[str, dict] = {}
tx_total = 0

for tx in iter_transactions(NDJSON_PATH, CLIENT_ID_FILTER):
    tx_total += 1
    key = profile_key(tx)
    prof = profiles.get(key)
    if prof is None:
        prof = {
            "id": profile_id(key),
            "profile_key": key,
            "txn_count": 0,
            "_amounts": [],
            **{f: tx.get(f) for f in PROFILE_FIELDS},
        }
        profiles[key] = prof
    prof["txn_count"] += 1
    amount = tx.get("amount_usd")
    if isinstance(amount, (int, float)) and len(prof["_amounts"]) < 50:
        prof["_amounts"].append(float(amount))

if tx_total == 0:
    raise ValueError(
        f"No transactions found for client_id={CLIENT_ID_FILTER} in {NDJSON_PATH.name}."
    )

for prof in profiles.values():
    amounts = prof.pop("_amounts")
    prof["median_amount_usd"] = round(statistics.median(amounts), 2) if amounts else None

print(f"Transactions scanned:   {tx_total:,}")
print(f"Distinct profiles:      {len(profiles):,}")
print(f"Compression:            {tx_total / max(len(profiles), 1):,.1f} transactions per LLM-classified profile")
print(f"Estimated LLM requests: {-(-len(profiles) // BATCH_SIZE)} (batch size {BATCH_SIZE})")


Transactions scanned:   30,672
Distinct profiles:      667
Compression:            46.0 transactions per LLM-classified profile
Estimated LLM requests: 17 (batch size 40)


## Guardrails around the LLM call

`categorize_core.categorize_with_cache_and_fallback` deliberately swallows LLM exceptions so
the pipeline never breaks. That is the right behavior for the user, but it makes failures
invisible — so this wrapper records them, enforces the cost cap, and lets the QA report state
plainly whether the run was degraded.

If no API key is present the notebook does **not** raise: it runs fully in deterministic
MCC-fallback mode and says so.


In [17]:
llm_state: dict = {
    "calls": 0,
    "failures": 0,
    "last_error": None,
    "enabled": bool(API_KEY),
    "disabled_reason": None,
}

raw_llm_call = None
if API_KEY:
    raw_llm_call = make_openai_llm_call(model=LLM_MODEL)
    print(f"LLM enabled: {LLM_MODEL}")
else:
    llm_state["disabled_reason"] = (
        "No LLM_API_KEY / OPENAI_API_KEY in environment. Add one to the project-root .env file."
    )
    print("WARNING:", llm_state["disabled_reason"])
    print("Running in deterministic MCC-fallback mode (100% coverage, no LLM labels).")


def guarded_llm_call(tx_batch: list[dict]) -> list[dict]:
    """Cost-capped, failure-recording wrapper around the raw OpenAI call."""
    if not llm_state["enabled"]:
        raise RuntimeError(llm_state["disabled_reason"] or "LLM disabled")
    if MAX_LLM_CALLS is not None and llm_state["calls"] >= MAX_LLM_CALLS:
        llm_state["enabled"] = False
        llm_state["disabled_reason"] = f"Cost cap reached (MAX_LLM_CALLS={MAX_LLM_CALLS})"
        raise RuntimeError(llm_state["disabled_reason"])

    llm_state["calls"] += 1
    try:
        return raw_llm_call(tx_batch)
    except Exception as exc:  # recorded, then re-raised into the core fallback path
        llm_state["failures"] += 1
        llm_state["last_error"] = f"{type(exc).__name__}: {exc}"
        raise


LLM enabled: gpt-5.2


## Categorize the profiles (cache -> LLM -> MCC fallback)

One extra guardrail lives here: when a batch is degraded (LLM failed or the cap tripped), the
resulting `mcc_fallback` labels are **evicted from the disk cache**. Otherwise a single run
without an API key would permanently cache rule-based labels and every future run would be a
cache hit that never reaches the LLM.


In [18]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache = Cache(str(CACHE_DIR))


def cache_key(tx: dict) -> str:
    """Recompute the key used inside categorize_core (kept in sync via CACHE_VERSION)."""
    return f"{CACHE_VERSION}:{transaction_fingerprint(tx, version=CACHE_VERSION)}"


# Pseudo-transactions: only the fields the prompt and the MCC fallback actually read.
profile_rows = [
    {
        "id": p["id"],
        "amount_usd": p["median_amount_usd"],
        "mcc_code": p["mcc_code"],
        "mcc_description": p["mcc_description"],
        "merchant_id": p["merchant_id"],
        "merchant_city": p["merchant_city"],
        "merchant_state": p["merchant_state"],
        "use_chip": "Online Transaction" if p["is_online"] else None,
    }
    for p in profiles.values()
]

assignments: dict[str, object] = {}
degraded_batches = 0

try:
    for start in range(0, len(profile_rows), BATCH_SIZE):
        batch = profile_rows[start : start + BATCH_SIZE]
        failures_before = llm_state["failures"]
        enabled_before = llm_state["enabled"]

        results = categorize_with_cache_and_fallback(
            batch,
            llm_call=guarded_llm_call,
            cache=cache,
            confidence_threshold=CONF_THRESHOLD,
            taxonomy=DEFAULT_TAXONOMY,
            cache_version=CACHE_VERSION,
        )

        degraded = (llm_state["failures"] > failures_before) or not enabled_before
        degraded_batches += int(degraded)
        by_id = {str(row["id"]): row for row in batch}

        for r in results:
            assignments[r.transaction_id] = r
            if degraded and r.source == "mcc_fallback":
                # do not persist a label produced while the LLM was unavailable
                cache.delete(cache_key(by_id[r.transaction_id]))

        done = min(start + BATCH_SIZE, len(profile_rows))
        print(
            f"  profiles {done:>5}/{len(profile_rows)}  "
            f"llm_calls={llm_state['calls']}  failures={llm_state['failures']}"
        )
finally:
    cache.close()

if llm_state["last_error"]:
    print("\nLast LLM error:", llm_state["last_error"])

print(f"\nProfiles categorized: {len(assignments):,} / {len(profile_rows):,}")
print("Source mix (profiles):", dict(Counter(r.source for r in assignments.values())))


  profiles    40/667  llm_calls=1  failures=0
  profiles    80/667  llm_calls=1  failures=0
  profiles   120/667  llm_calls=1  failures=0
  profiles   160/667  llm_calls=1  failures=0
  profiles   200/667  llm_calls=1  failures=0
  profiles   240/667  llm_calls=1  failures=0
  profiles   280/667  llm_calls=1  failures=0
  profiles   320/667  llm_calls=1  failures=0
  profiles   360/667  llm_calls=1  failures=0
  profiles   400/667  llm_calls=1  failures=0
  profiles   440/667  llm_calls=1  failures=0
  profiles   480/667  llm_calls=1  failures=0
  profiles   520/667  llm_calls=1  failures=0
  profiles   560/667  llm_calls=1  failures=0
  profiles   600/667  llm_calls=1  failures=0
  profiles   640/667  llm_calls=1  failures=0
  profiles   667/667  llm_calls=1  failures=0

Profiles categorized: 667 / 667
Source mix (profiles): {'llm': 37, 'low_confidence_fallback': 3, 'mcc_fallback': 627}


## Pass 2 — write the per-transaction mapping artifact

Every transaction is joined back to its profile label, so coverage is 100% by construction.
The QA metrics are computed in the same pass: `mcc_agreement_pct` is the share of LLM labels
that match the deterministic MCC rule wherever that rule has an opinion — with no ground-truth
labels available, agreement + fallback rate is the evaluation signal.


In [19]:
if OUT_CATEGORIES.exists():
    OUT_CATEGORIES.unlink()

source_counts: Counter = Counter()
category_counts: Counter = Counter()
written = 0
unmapped = 0
agree = 0
comparable = 0

with OUT_CATEGORIES.open("w", encoding="utf-8") as out:
    for tx in iter_transactions(NDJSON_PATH, CLIENT_ID_FILTER):
        prof = profiles.get(profile_key(tx))
        result = assignments.get(prof["id"]) if prof is not None else None
        if result is None:
            unmapped += 1
            continue

        row = {
            "id": tx.get("id"),
            "client_id": tx.get("client_id"),
            "transaction_dt": tx.get("transaction_dt"),
            "amount_usd": tx.get("amount_usd"),
            "mcc_code": tx.get("mcc_code"),
            "mcc_description": tx.get("mcc_description"),
            "category": result.category_final,
            "source": result.source,
            "confidence": result.confidence,
            "category_llm": result.category_llm,
            "model": LLM_MODEL if result.source == "llm" else None,
            "profile_id": prof["id"],
        }
        out.write(json.dumps(row, ensure_ascii=False) + "\n")
        written += 1

        source_counts[result.source] += 1
        category_counts[result.category_final] += 1

        rule_category = mcc_to_category(tx.get("mcc_code"), tx.get("mcc_description"))
        if result.category_llm and rule_category != "Other/Uncategorized":
            comparable += 1
            agree += int(result.category_llm == rule_category)

OUT_PROFILES.write_text(
    json.dumps(
        {
            "client_id": CLIENT_ID_FILTER,
            "profile_fields": list(PROFILE_FIELDS),
            "profiles": [
                {**p, "category": getattr(assignments.get(p["id"]), "category_final", None),
                 "source": getattr(assignments.get(p["id"]), "source", None)}
                for p in profiles.values()
            ],
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Rows written: {written:,}  (unmapped: {unmapped})")
print("Wrote:", OUT_CATEGORIES)
print("Wrote:", OUT_PROFILES)


Rows written: 30,672  (unmapped: 0)
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transaction_categories_1696.jsonl
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/merchant_profiles_1696.json


## QA report artifact

This is the evidence the evaluation checklist asks for: coverage, LLM vs fallback mix, agreement with the deterministic rule, and the cost counters that prove the guardrails held.


In [20]:
degraded_run = bool(llm_state["failures"]) or not llm_state["enabled"]

report = {
    "client_id": CLIENT_ID_FILTER,
    "model": LLM_MODEL,
    "taxonomy": list(DEFAULT_TAXONOMY),
    "confidence_threshold": CONF_THRESHOLD,
    "batch_size": BATCH_SIZE,
    "max_llm_calls": MAX_LLM_CALLS,
    "cache_version": CACHE_VERSION,
    "transactions_scanned": tx_total,
    "transactions_categorized": written,
    "transactions_unmapped": unmapped,
    "coverage_pct": round(100.0 * written / tx_total, 4) if tx_total else 0.0,
    "distinct_profiles": len(profiles),
    "transactions_per_profile": round(tx_total / max(len(profiles), 1), 2),
    "llm_calls_made": llm_state["calls"],
    "llm_call_failures": llm_state["failures"],
    "llm_last_error": llm_state["last_error"],
    "llm_enabled_at_end": llm_state["enabled"],
    "degraded_run": degraded_run,
    "degraded_reason": llm_state["disabled_reason"],
    "degraded_batches": degraded_batches,
    "source_mix_transactions": dict(source_counts),
    "source_mix_pct": {
        k: round(100.0 * v / max(written, 1), 2) for k, v in source_counts.items()
    },
    "mcc_agreement_pct": round(100.0 * agree / comparable, 2) if comparable else None,
    "mcc_agreement_sample": comparable,
    "category_mix_transactions": dict(category_counts.most_common()),
    "outputs": {
        "categories": str(OUT_CATEGORIES.relative_to(ROOT)),
        "profiles": str(OUT_PROFILES.relative_to(ROOT)),
        "report": str(OUT_REPORT.relative_to(ROOT)),
    },
}

OUT_REPORT.write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")

print(json.dumps({k: v for k, v in report.items() if k != "category_mix_transactions"}, indent=2))
print("\nWrote:", OUT_REPORT)

if degraded_run:
    print(
        "\nWARNING: degraded run — some or all labels came from the deterministic MCC rule, "
        "not the LLM. Fix the cause and rerun; cached fallback labels were not persisted."
    )


{
  "client_id": 1696,
  "model": "gpt-5.2",
  "taxonomy": [
    "Dining",
    "Groceries",
    "Utilities",
    "Transportation",
    "Entertainment",
    "Shopping",
    "Travel",
    "Housing",
    "Healthcare",
    "Education",
    "Income",
    "Transfers",
    "Subscriptions",
    "Fees & Interest",
    "Cash Withdrawal",
    "Other/Uncategorized"
  ],
  "confidence_threshold": 0.6,
  "batch_size": 40,
  "max_llm_calls": 1,
  "cache_version": "v1",
  "transactions_scanned": 30672,
  "transactions_categorized": 30672,
  "transactions_unmapped": 0,
  "coverage_pct": 100.0,
  "distinct_profiles": 667,
  "transactions_per_profile": 45.99,
  "llm_calls_made": 1,
  "llm_call_failures": 0,
  "llm_last_error": null,
  "llm_enabled_at_end": false,
  "degraded_run": true,
  "degraded_reason": "Cost cap reached (MAX_LLM_CALLS=1)",
  "degraded_batches": 14,
  "source_mix_transactions": {
    "llm": 25428,
    "low_confidence_fallback": 50,
    "mcc_fallback": 5194
  },
  "source_mix_pct": {


## Preview

Sanity check on the mapping artifact before analytics consumes it.


In [21]:
preview = pd.read_json(OUT_CATEGORIES, lines=True, nrows=2000)
display(preview.head(15))

print("\nCategory mix (top 10, all rows):")
for category, count in Counter(category_counts).most_common(10):
    print(f"  {category:<22} {count:>8,}  ({100.0 * count / max(written, 1):5.1f}%)")


,id,client_id,transaction_dt,amount_usd,mcc_code,mcc_description,category,source,confidence,category_llm,model,profile_id
0,7475539,1696,2010-01-01 05:38:00,4.02,5812,Eating Places and Restaurants,Dining,llm,0.90,Dining,gpt-5.2,pe06ac9de0ba20a81
1,7475586,1696,2010-01-01 06:03:00,9.68,4784,Tolls and Bridge Fees,Transportation,llm,0.85,Transportation,gpt-5.2,p3e24ec197ada92e7
2,7475755,1696,2010-01-01 06:53:00,3.41,5411,"Grocery Stores, Supermarkets",Groceries,llm,0.90,Groceries,gpt-5.2,p0aa3598500fed92e
3,7477220,1696,2010-01-01 12:18:00,9.94,4784,Tolls and Bridge Fees,Transportation,llm,0.85,Transportation,gpt-5.2,p8c8f3bef51cc685d
4,7477493,1696,2010-01-01 13:11:00,-89.00,5541,Service Stations,Transportation,llm,0.75,Transportation,gpt-5.2,p40901ca87ce89b1d
5,7477530,1696,2010-01-01 13:18:00,89.00,5541,Service Stations,Transportation,llm,0.75,Transportation,gpt-5.2,p40901ca87ce89b1d
6,7477584,1696,2010-01-01 13:29:00,149.43,5541,Service Stations,Transportation,llm,0.75,Transportation,gpt-5.2,p40901ca87ce89b1d
7,7478016,1696,2010-01-01 15:18:00,9.15,4784,Tolls and Bridge Fees,Transportation,llm,0.85,Transportation,gpt-5.2,p3e24ec197ada92e7
8,7479654,1696,2010-01-02 06:06:00,4.03,4784,Tolls and Bridge Fees,Transportation,llm,0.85,Transportation,gpt-5.2,p3e24ec197ada92e7
9,7480705,1696,2010-01-02 10:57:00,93.42,5411,"Grocery Stores, Supermarkets",Groceries,llm,0.90,Groceries,gpt-5.2,p0aa3598500fed92e



Category mix (top 10, all rows):
  Transportation           18,019  ( 58.7%)
  Groceries                 9,039  ( 29.5%)
  Dining                    1,062  (  3.5%)
  Other/Uncategorized         847  (  2.8%)
  Shopping                    823  (  2.7%)
  Utilities                   370  (  1.2%)
  Healthcare                  209  (  0.7%)
  Transfers                   200  (  0.7%)
  Entertainment               103  (  0.3%)
